# Backbone benchmark — thin demo

The full backbone-benchmark pipeline now lives in the codebase under `src/`:

- **Feature extractor** — [src/features/cnn.py](../src/features/cnn.py) (`CNNFeatureExtractor`)
- **Embedder** — [src/embedders/spatial_attention.py](../src/embedders/spatial_attention.py) (`SpatialAttentionEmbedder`)
- **Decoder** — [src/decoders/pose_slot.py](../src/decoders/pose_slot.py) (`PoseSlotDecoder`)
- **Set-prediction loss / OKS / Hungarian** — [src/training/losses.py](../src/training/losses.py)
- **Set-format dataset & collate** — [src/data/set_pose_dataset.py](../src/data/set_pose_dataset.py)
- **Lightning module** — [src/training/set_prediction_module.py](../src/training/set_prediction_module.py)
- **Hydra configs** — [conf/pose_estimator/cnn_set_prediction.yaml](../conf/pose_estimator/cnn_set_prediction.yaml), [conf/training_module/set_prediction.yaml](../conf/training_module/set_prediction.yaml), [conf/data/crowdpose_set_prediction.yaml](../conf/data/crowdpose_set_prediction.yaml)

To run a full benchmark, use the Hydra script (no notebook execution required):

```bash
python scripts/benchmark_backbone.py
python scripts/benchmark_backbone.py training_module.max_epochs=100 data.batch_size=16
python scripts/benchmark_backbone.py skip_training=true \
    training_module.resume=results/backbone_benchmark/.../checkpoints/last.ckpt
```

Coordinate / contract conventions are documented in [docs/assumptions.md](../docs/assumptions.md).

The remainder of this notebook is a small smoke demo: compose the Hydra config, build the model, run one forward pass, and compute one set-prediction loss.

In [58]:
"""Compose the Hydra config and build a SetPredictionTrainingModule."""

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir

from src.data import DataConfig
from src.pose_estimator import PoseEstimatorConfig
from src.training import TrainingModuleConfig

with initialize_config_dir(version_base="1.3", config_dir=str(PROJECT_ROOT / "conf")):
    cfg = compose(config_name="benchmark_backbone")

pose_cfg = PoseEstimatorConfig.from_hydra_config(cfg.pose_estimator)
tm_cfg   = TrainingModuleConfig.from_hydra_config(cfg.training_module)
data_cfg = DataConfig.from_hydra_config(cfg.data)

pose_estimator = pose_cfg.build()
data_module    = data_cfg.build(pin_memory=False)
module         = tm_cfg.build(pose_estimator, data=data_module)

n_params = sum(p.numel() for p in pose_estimator.parameters() if p.requires_grad)
print(f"FeatureExtractor : {type(pose_estimator.feature_extractor).__name__}")
print(f"FeatureEmbedder  : {type(pose_estimator.feature_embedder).__name__}")
print(f"Decoder          : {type(pose_estimator.decoder).__name__}")
print(f"PoseEstimator    : {n_params:,} trainable parameters")
print(f"TrainingModule   : {type(module).__name__}")


loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.22s)
creating index...
index created!
FeatureExtractor : CNNFeatureExtractor
FeatureEmbedder  : SpatialAttentionEmbedder
Decoder          : PoseSlotDecoder
PoseEstimator    : 1,452,348 trainable parameters
TrainingModule   : SetPredictionTrainingModule


In [61]:
"""Smoke forward pass on a synthetic set-format batch.

Confirms that the wired-up pipeline runs end-to-end without touching the
real CrowdPose dataset (avoids slow notebook startup).
"""

import torch

from src.training.losses import set_prediction_loss
from src.metrics import CROWDPOSE_SIGMAS

sigmas = torch.from_numpy(CROWDPOSE_SIGMAS).float()

B, P, K = 2, 3, 14
H, W = int(cfg.data.image_h), int(cfg.data.image_w)
device = torch.device("cpu")

batch = {
    "image":         torch.rand(B, 3, H, W, device=device),
    "gt_keypoints":  torch.rand(B, P, K, 2, device=device) * 2 - 1,
    "gt_labeled":    torch.ones(B, P, K, dtype=torch.bool, device=device),
    "gt_visible":    torch.ones(B, P, K, dtype=torch.bool, device=device),
    "person_scale":  torch.full((B, P), 0.3, device=device),
    "person_mask":   torch.ones(B, P, dtype=torch.bool, device=device),
}

module.eval()
with torch.no_grad():
    pred = module.pose_estimator(image=batch["image"])

print("Decoder outputs:")
for k, v in pred.items():
    print(f"  {k:18s} {tuple(v.shape)}  dtype={v.dtype}")

loss_terms = set_prediction_loss(
    pred, batch, sigmas=sigmas, weights=module.weights,
)
print()
print("Set-prediction loss terms:")
for k, v in loss_terms.items():
    print(f"  {k:14s} {v.item():.4f}")


Decoder outputs:
  keypoints          (2, 20, 14, 2)  dtype=torch.float32
  scores             (2, 20)  dtype=torch.float32
  scores_logits      (2, 20)  dtype=torch.float32
  keypoint_scores    (2, 20, 14)  dtype=torch.float32
  vis_logits         (2, 20, 14)  dtype=torch.float32

Set-prediction loss terms:
  total          6.9944
  oks            0.9818
  l1             1.0023
  obj            0.6643
  vis            0.6739
  mean_oks       0.0182
  matched        6.0000
